# Libraries

In [1]:
# 0. Clean slate — remove existing copies before reinstalling
!pip uninstall -y torch torchvision torchaudio torchao

!pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 torchao==0.16.0 --index-url https://download.pytorch.org/whl/cu128

# 3. Evaluation Framework
!pip install -q "lm_eval[hf]"

# 4. Compression Tools
!pip install -q gptqmodel==6.0.3
!pip install -q optimum

# 5. Utilities
!pip install -q langdetect --break-system-packages  # for IFEval Task

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 100.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.2 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolv

In [2]:
!pip install -q "cuda-core==0.3.*" "numba-cuda[cu12]<0.23.0,>=0.22.2" "numba<0.62.0,>=0.60.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 32.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 96.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 45.6 MB/s eta 0:00:00:00:0100:01


In [3]:
!pip install -q "kernels<0.15" --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 2.1 MB/s eta 0:00:00


In [4]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file
import traceback

# Global Variables

In [5]:
# --- Identity ---
user_name = "Abdelrahmanemam01"
email     = "abdulrahmanemam01@gmail.com"
repo      = "MahmoudOsama20/EdgeCompress"  # Fixed

# --- Model ---
model      = "Phi-4-mini-instruct"
MODEL_NAME       = "Phi-4-mini-instruct-GPTQ"
MODEL_PRETRAINED = "EdgeCompress01/Phi-4-mini-instruct-GPTQ-4bit"
MODEL_PRETRAINED_FIX = MODEL_PRETRAINED.replace("/", "__")

# --- Reproducibility ---
SEED = 42

# --- Environment ---
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [6]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    max_gen_toks:      Optional[int] = None    

# Functions

In [7]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [8]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface login

In [9]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# GitHub login

In [10]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

Logging into GitHub...


# Evaluations

**Configurations**

In [11]:
# Task definition
TASKS = [
    # Instruction Following
     Task("ifeval",       "ifeval",                    "instruction_following",batch_size=16,  max_gen_toks=1024),

    # Knowledge
     Task("mmlu",         "mmlu",                      "knowledge",           num_fewshot=5, batch_size=2,  limit=1400),

    # Math
     Task("gsm8k",        "gsm8k_cot",                 "math",                num_fewshot=8, batch_size=16,  max_gen_toks=1024),
]

In [12]:
local_path = snapshot_download(
    repo_id=MODEL_PRETRAINED,
    local_dir="/kaggle/working/model"
)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [13]:
src = local_path
dst = "/kaggle/working/model_fused"
os.makedirs(dst, exist_ok=True)

# Copy all non-weight files
for f in os.listdir(src):
    src_path = os.path.join(src, f)
    dst_path = os.path.join(dst, f)
    
    if not f.endswith(".safetensors") and os.path.isfile(src_path):  # skip dirs
        shutil.copy2(src_path, dst_path)

weights = load_file(f"{src}/model.safetensors")
new_weights = {}
processed = set()

for key in weights:
    if key in processed:
        continue
    if ".mlp.gate_proj." in key:
        suffix = key.split(".mlp.gate_proj.")[-1]
        prefix = key.split(".mlp.gate_proj.")[0]
        up_key = f"{prefix}.mlp.up_proj.{suffix}"
        fused_key = f"{prefix}.mlp.gate_up_proj.{suffix}"

        gate_t = weights[key]
        up_t   = weights[up_key]

        if suffix == "g_idx":
            # g_idx indexes input groups — same for both, keep one
            new_weights[fused_key] = gate_t
        elif suffix == "qweight":
            # qweight: [in_features//8, out_features] → concat on out dim (dim=1)
            new_weights[fused_key] = torch.cat([gate_t, up_t], dim=1)
        else:
            # scales, qzeros: [n_groups, out_features] → concat on out dim (dim=1)
            new_weights[fused_key] = torch.cat([gate_t, up_t], dim=1)

        processed.update([key, up_key])
    elif ".mlp.up_proj." in key:
        processed.add(key)  # already fused above
    else:
        new_weights[key] = weights[key]
        processed.add(key)

save_file(new_weights, f"{dst}/model.safetensors")
print(f"Done. {len(weights)} → {len(new_weights)} tensors")

Done. 706 → 578 tensors


In [14]:
# Model args
MODEL_ARGS = ",".join([
    f"pretrained={dst}",
    "tokenizer=microsoft/Phi-4-mini-instruct",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
])

results_dir = f"evaluation_results/{MODEL_NAME}"

In [15]:
print(results_dir)

evaluation_results/Phi-4-mini-instruct-GPTQ


**Evaluation loop**

In [ ]:
try:
    for t in TASKS:
        print(f"\n[{t.category.upper()}] {t.key}")
        cmd = [
            "lm_eval",
            "--model",       "hf",
            "--model_args",  MODEL_ARGS,
            "--tasks",       t.name,
            "--num_fewshot", str(t.num_fewshot),
            "--batch_size",  str(t.batch_size),
            "--seed",        str(SEED),
            "--verbosity", "DEBUG",
            "--output_path", f"{results_dir}/{t.key}",
        ]
    
        if t.limit is not None:
            cmd += ["--limit", str(t.limit)]
            
        if t.apply_chat_template:
                cmd.append("--apply_chat_template")
            
        if t.max_gen_toks is not None:                             
            cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]
    
        subprocess.run(cmd, check=True)
        
        # Push To GitHub
        target_dir_in_repo = f"Evaluations/BenchmarkEvaluations/{model}/{MODEL_NAME}/results/{t.key}"
        source_folder = f"{results_dir}/{t.key}/__kaggle__working__model_fused"  # The Kaggle folder you want to upload
        remote_url = f"https://{token}@github.com/{repo}.git"
        
        # A temporary workspace in Kaggle to handle the cloning
        temp_repo_dir = "/kaggle/working/temp_git_repo" 
        # ---------------------
    
        # 2. Clean up any previous runs and clone the existing repository
        os.system(f"rm -rf {temp_repo_dir}")
        os.system(f"git clone {remote_url} {temp_repo_dir}")
        
        # 3. Create the target directory inside the cloned repo
        full_target_path = f"{temp_repo_dir}/{target_dir_in_repo}"
        os.makedirs(full_target_path, exist_ok=True)
        
        # 4. Copy the contents of your Kaggle folder into the target GitHub folder
        # (Using cp -r to recursively copy all files and subfolders)
        os.system(f"cp -r {source_folder}/* {full_target_path}/")
        
        # 5. Configure, commit, and push the changes
        os.system(f"""
            git -C {temp_repo_dir} config user.email "{email}" && \
            git -C {temp_repo_dir} config user.name "{user_name}" && \
            git -C {temp_repo_dir} add . && \
            git -C {temp_repo_dir} commit -m "Add Kaggle results to {target_dir_in_repo}" && \
            git -C {temp_repo_dir} push origin main
        """)
        
        print(f"Successfully pushed to {repo} inside the '{target_dir_in_repo}' folder!")
        
        torch.cuda.empty_cache()
        gc.collect()
        print("Done")
except Exception as e:
    traceback.print_exc()


[INSTRUCTION_FOLLOWING] ifeval


2026-05-31:12:58:48 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-05-31:12:59:01 INFO     [_cli.run:388] Selected Tasks: ['ifeval']
2026-05-31:12:59:01 INFO     [evaluator:162] Setting verbosity through simple_evaluate is deprecated.
2026-05-31:12:59:03 INFO     [evaluator:214] Setting random seed to 42 | Setting numpy seed to 42 | Setting torch manual seed to 42 | Setting fewshot manual seed to 42
2026-05-31:12:59:03 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-05-31:12:59:03 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/working/model_fused', 'tokenizer': 'microsoft/Phi-4-mini-instruct', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}
2026-05-31:12:59:10 INFO     [models.huggingface:306] Using `accelerate launch` or `parallelize=True`, devi


{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pr

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
INFO:httpx:HTTP Request: GET https://huggingface.co/api/organizations/kernels-community/overview "HTTP/1.1 200 OK"
/usr/local/lib/python3.12/dist-packages/kernels/utils.py:391: FutureWarning: Future versions of `kernels` (>=0.15) will require specifying a kernel version or revision. See: https://huggingface.co/docs/kernels/migration
  revision = select_revision_or_version(repo_id, revision=revision, version=version)
INFO:httpx:HTTP Request: HEAD https://huggingface.co/kernels/kernels-community/quantization-gptq/resolve/main/kernel-status.toml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/kernels/kernels-community/quantization-gptq/tree/main/build?recursive=false&expand=false "HTTP/1.1 200 OK"
Failed to load CPU gemm_4bit kernel from `kernels-community/quantization-gptq`: cannot import name 'package_name_from_

{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 6.0.3
Transformers : 5.9.0
Torch        : 2.10.0+cu128
Triton       : 3.6.0
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  Kernel: Auto-selection: adding candidate `TritonV2QuantLinear`           
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"><

Loading weights: 100%|██████████| 578/578 [00:01<00:00, 441.22it/s]


{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  QuantizeConfig: offload_to_disk_path auto set to `./gptqmodel_offload/pxhwvnby-pkljwdmk/`
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  Format: Converting `format` from `FORMAT.GPTQ` to internal `FORMAT.GPTQ_V2`.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  Format: Converting GPTQ v1 to v2                                         
{

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/google/IFEval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/google/IFEval/966cd89545d6b6acfd7638bc708b98261ca58e84/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/google/IFEval/966cd89545d6b6acfd7638bc708b98261ca58e84/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/google/IFEval/resolve/966cd89545d6b6acfd7638bc708b98261ca58e84/IFEval.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/google/IFEval/google/IFEval.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/google/IFEval/revision/966cd89545d6b6acfd7638bc708b98261ca58e84 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/google/IFEval/resolv

# Save reports

In [ ]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)